# 04 L2/3 Normalized Covariance Spectrum

Standalone layer that compares full normalized covariance with the L2/3 submatrix.

In [ ]:
from pathlib import Path
import sys

module_dir = Path.cwd()
if not (module_dir / "functional_analysis_utils.py").exists() and (Path.cwd() / "funconn-analysis" / "functional_analysis_utils.py").exists():
    module_dir = Path.cwd() / "funconn-analysis"
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from functional_analysis_utils import (
    RUN_DATE,
    eigenspectrum_desc,
    eigenspectrum_summary,
    fit_power_law_positive,
    functional_paths,
    matrix_abs_limit,
    plot_power_law_fit,
    plot_positive_eigenvalues,
    plot_subsampled_eigenspectrum,
    read_matrix_csv,
    save_eigenspectrum_csv,
    save_matrix,
    sorted_diagonal_spectrum,
)

sns.set_theme(style="white", context="notebook")
paths = functional_paths()
CORR_DIR = paths["correlation"]
EIGEN_DIR = paths["coupling_eigen"]
LAYER_DIR = paths["layer_annotation"]
print("Functional output:", paths["functional_base"])


In [ ]:
normalized_cov_df = read_matrix_csv(CORR_DIR / "V1_col_7_4_activity_normalized_covariance_matrix.csv")
layer_annotations = pd.read_csv(LAYER_DIR / "V1_col_7_4_root_id_layer_annotations.csv")

unit_ids = normalized_cov_df.index.astype(int).to_numpy()
layer_info = pd.DataFrame({"unit_id": unit_ids}).merge(
    layer_annotations[["unit_id", "pt_root_id", "allen_slanted_cell_type", "allen_slanted_layer"]],
    on="unit_id",
    how="left",
)
layer_info["allen_slanted_layer"] = layer_info["allen_slanted_layer"].fillna("Missing")
layer_info["allen_slanted_cell_type"] = layer_info["allen_slanted_cell_type"].fillna("Missing")

l23_unit_ids = layer_info.loc[layer_info["allen_slanted_layer"].eq("L2/3"), "unit_id"].to_numpy(dtype=int)
l23_unit_ids = np.asarray([unit_id for unit_id in unit_ids if unit_id in set(l23_unit_ids)], dtype=int)
if len(l23_unit_ids) < 3:
    raise ValueError(f"Need at least 3 L2/3 neurons, found {len(l23_unit_ids)}.")

full_norm_cov_df = normalized_cov_df.loc[unit_ids, unit_ids]
l23_norm_cov_df = normalized_cov_df.loc[l23_unit_ids, l23_unit_ids]
full_norm_cov_matrix = full_norm_cov_df.to_numpy(dtype=float)
l23_norm_cov_matrix = l23_norm_cov_df.to_numpy(dtype=float)

print("Full N:", len(unit_ids), "Tr(C)/N:", np.trace(full_norm_cov_matrix) / full_norm_cov_matrix.shape[0])
print("L2/3 N:", len(l23_unit_ids), "Trace/N:", np.trace(l23_norm_cov_matrix) / l23_norm_cov_matrix.shape[0])


In [ ]:
l23_norm_cov_csv = CORR_DIR / "V1_col_7_4_activity_normalized_covariance_matrix_L23.csv"
save_matrix(l23_norm_cov_df, l23_norm_cov_csv)

full_norm_cov_evals = eigenspectrum_desc(full_norm_cov_matrix)
l23_norm_cov_evals = eigenspectrum_desc(l23_norm_cov_matrix)
fit_full_norm_cov = fit_power_law_positive(full_norm_cov_evals, num_top_eigenvalues=min(100, len(full_norm_cov_evals)))
fit_l23_norm_cov = fit_power_law_positive(l23_norm_cov_evals, num_top_eigenvalues=min(100, len(l23_norm_cov_evals)))

l23_norm_cov_eigenspectrum_df = save_eigenspectrum_csv(
    l23_norm_cov_evals,
    EIGEN_DIR / "V1_col_7_4_activity_normalized_covariance_L23_eigenspectrum.csv",
)
summary_df = pd.DataFrame([
    eigenspectrum_summary("full_normalized_covariance", full_norm_cov_evals, fit_full_norm_cov),
    eigenspectrum_summary("L2/3_normalized_covariance", l23_norm_cov_evals, fit_l23_norm_cov),
])
summary_csv = EIGEN_DIR / "V1_col_7_4_activity_normalized_covariance_L23_vs_full_eigenspectrum_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)
display(summary_df)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.8), dpi=150)
plot_positive_eigenvalues(ax, full_norm_cov_evals, "Full normalized covariance", "black")
plot_positive_eigenvalues(ax, l23_norm_cov_evals, f"L2/3 normalized covariance (N={len(l23_unit_ids)})", "#2f7f8f")
plot_power_law_fit(ax, fit_full_norm_cov, "Full fit", "black", linewidth=1.1)
plot_power_law_fit(ax, fit_l23_norm_cov, "L2/3 fit", "#d54b3d", linewidth=1.1)
ax.set_xlabel("Rank (r/N)")
ax.set_ylabel("Eigenvalue")
ax.set_title("Full vs L2/3 Normalized Activity Covariance Eigenspectrum")
ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
png_path = EIGEN_DIR / "V1_col_7_4_activity_normalized_covariance_L23_vs_full_eigenspectrum.png"
fig.savefig(png_path, bbox_inches="tight")
plt.show()
print("Saved:", png_path)
display(l23_norm_cov_eigenspectrum_df.head(10))
